# NutriScan v2 — YOLOv8s Retraining with Filtered Classes

**Authors** Yu Cao, Na Yin, Fan Zhang

**Date:** April 2026

**Runtime:** GPU (A100 recommended)  
**Drive layout:**
```
MyDrive/nutriscan/
├── dataset256.zip          ← source archive (you provide)
└── v2/
    ├── uec_yolo_v2/        ← converted YOLO dataset (cached here)
    ├── data_v2.yaml
    ├── classes_v2.txt
    ├── best_v2.pt
    ├── last_v2.pt
    └── results_v2.csv
```

**Steps:**
1. Mount Google Drive and set paths
2. Install ultralytics
3. Extract `dataset256.zip` (skips if already extracted)
4. Convert UEC FOOD-256 → YOLO format with class filtering (skips if Drive cache exists)
5. Write `data.yaml`
6. Train YOLOv8s for 100 epochs
7. Save weights + metrics to Drive
8. Validate and report per-class mAP50

## 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ── Path constants used throughout the notebook ────────────────────────────
DRIVE_DIR  = Path('/content/drive/MyDrive/nutriscan')   # existing base folder
SAVE_DIR   = Path('/content/drive/MyDrive/nutriscan/v2') # all v2 outputs go here
YOLO_DRIVE = SAVE_DIR / 'uec_yolo_v2'                   # converted dataset on Drive

os.makedirs(str(SAVE_DIR), exist_ok=True)

print(f'Drive base : {DRIVE_DIR}')
print(f'Save dir   : {SAVE_DIR}')
print(f'Data dir   : {YOLO_DRIVE}')

Mounted at /content/drive
Drive base : /content/drive/MyDrive/nutriscan
Save dir   : /content/drive/MyDrive/nutriscan/v2
Data dir   : /content/drive/MyDrive/nutriscan/v2/uec_yolo_v2


## 2 — Install ultralytics

In [2]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Setup complete ✅ (12 CPUs, 83.5 GB RAM, 42.9/235.7 GB disk)


## 3 — Extract UEC FOOD-256

Place `dataset256.zip` in `MyDrive/nutriscan/` before running.

The zip contains 256 numbered sub-folders. Each sub-folder has `.jpg` images
and a `bb_info.txt` file (space-separated, 1 header row):
```
img    x1    y1    x2    y2
1      52    36    253   317
```
This cell is skipped if `/content/UECFOOD256` already exists.

In [3]:
import zipfile
import pathlib

DATASET_ROOT = pathlib.Path('/content/UECFOOD256')
ARCHIVE_PATH = DRIVE_DIR / 'dataset256.zip'

if DATASET_ROOT.exists():
    print(f'Dataset already extracted at {DATASET_ROOT} — skipping.')
else:
    if not ARCHIVE_PATH.exists():
        raise FileNotFoundError(
            f'Archive not found: {ARCHIVE_PATH}\n'
            'Download dataset256.zip from http://foodcam.mobi/dataset256.html '
            f'and upload it to {DRIVE_DIR}/'
        )
    print(f'Extracting {ARCHIVE_PATH} ...')
    with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
        zf.extractall('/content/')
    print('Extraction complete.')

    # Handle zips that extract to a differently-named root folder
    if not DATASET_ROOT.exists():
        candidates = list(pathlib.Path('/content').rglob('bb_info.txt'))
        if not candidates:
            raise RuntimeError('bb_info.txt not found after extraction — check the zip structure.')
        for candidate in candidates:
            parent = candidate.parent.parent  # e.g. /content/SomeName/1/ -> /content/SomeName
            if parent != DATASET_ROOT:
                print(f'Renaming {parent} → {DATASET_ROOT}')
                parent.rename(DATASET_ROOT)
            break

# Verify category folders
cat_dirs = sorted(
    [d for d in DATASET_ROOT.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda p: int(p.name),
)
print(f'Found {len(cat_dirs)} category folders (expected 256).')
if len(cat_dirs) < 256:
    print('WARNING: fewer than 256 folders — zip may be incomplete.')

Extracting /content/drive/MyDrive/nutriscan/dataset256.zip ...
Extraction complete.
Found 256 category folders (expected 256).


## 4 — Convert `bb_info.txt` Annotations → YOLO Format (with class filtering)

**Class filtering:** classes in `SKIP_CLASSES` are excluded because they either
have too few samples (< ~100 images) or scored mAP50 < 0.55 in a prior training run.

**Drive cache:** if `YOLO_DRIVE` already exists on Drive the conversion step is
skipped entirely — the cached dataset is used directly. This saves 10-15 minutes
on subsequent runs.

**Conversion details:**
- UEC pixel coords `(x1,y1,x2,y2)` → normalised YOLO `cx cy w h`
- 85 / 15 train / val random split (seed 42)
- Images with no valid boxes are discarded
- Degenerate boxes (`x2 <= x1` or `y2 <= y1`) are skipped

In [4]:
import random
import shutil
from pathlib import Path
from PIL import Image

# ── Classes to skip ────────────────────────────────────────────────────────
# Reason A: estimated total sample count < 100
# Reason B: mAP50 < 0.55 in previous training run
SKIP_CLASSES = {
    # Reason A — too few samples
    'xiao_long_bao', 'yellow_curry', 'Fried_spring_rolls',
    'coconut_milk_soup', 'gulai', 'minced_pork_rice',
    'Crispy_Noodles', 'Chicken_Rice_Curry_With_Coconut',
    'sushi_bowl', 'shrimp_with_chill_source', 'laulau',
    'Egg_Noodle_In_Chicken_Yellow_Curry', 'pho',
    'ball_shaped_bun_with_pork', 'mie_ayam',

    # Reason B — poor mAP50 < 0.55
    'meat_loaf', 'grilled_eggplant', 'tanmen',
    'pork_belly', 'fried_fish', 'beef_in_oyster_sauce',
    'pork_miso_soup', 'ayam_goreng', 'cold_tofu',
    'chip_butty', 'noodles_with_fish_curry', 'rice',
    'green_salad', 'Rice_crispy_pork', 'french_fries',
    'sauteed_vegetables', 'sausage', 'nasi_campur',
    'chicken_cutlet', 'boiled_fish', 'teriyaki_grilled_fish',
    'fried_chicken', 'tempura', 'grilled_salmon',
    'salmon_meuniere', 'sirloin_cutlet', 'omelet',
    'ganmodoki', 'miso_soup', 'simmered_pork',
    'nanbanzuke', 'stewed_pork_leg', 'Pork_with_lemon',
    'laksa', 'chow_mein', 'oxtail_soup', 'waffle',
    'roast_duck', 'rice_ball', 'pork_loin_cutlet',
    'pork_fillet_cutlet', 'ham_cutlet', 'minced_meat_cutlet',
    'egg_sunny-side_up', 'sukiyaki', 'fried_noodle',
}

# ── Drive cache check ──────────────────────────────────────────────────────
if (YOLO_DRIVE / 'images' / 'train').exists():
    print('Dataset already exists in Drive — skipping conversion.')
    print(f'Using cached dataset at: {YOLO_DRIVE}')
    YOLO_ROOT = YOLO_DRIVE

    # Re-derive class_names from data.yaml if present, else from classes_v2.txt
    import yaml
    yaml_path = YOLO_ROOT / 'data.yaml'
    classes_txt = SAVE_DIR / 'classes_v2.txt'
    if yaml_path.exists():
        with open(yaml_path) as f:
            _d = yaml.safe_load(f)
        class_names = _d['names']
    elif classes_txt.exists():
        class_names = [l.strip() for l in classes_txt.read_text().splitlines() if l.strip()]
    else:
        class_names = []  # will be rebuilt in cell 5
    print(f'Loaded {len(class_names)} class names.')

else:
    # ── Local working directory for conversion ─────────────────────────────
    YOLO_LOCAL = Path('/content/uec_yolo_v2')
    for split in ('train', 'val'):
        (YOLO_LOCAL / 'images' / split).mkdir(parents=True, exist_ok=True)
        (YOLO_LOCAL / 'labels' / split).mkdir(parents=True, exist_ok=True)

    # ── Build class name map from category.txt ─────────────────────────────
    class_names: list[str] = []
    cat_id_map: dict[str, int] = {}   # folder name → YOLO class index
    skipped_class_names: list[str] = []

    root_cat_file = DATASET_ROOT / 'category.txt'
    name_by_folder: dict[str, str] = {}
    if root_cat_file.exists():
        for line in root_cat_file.read_text(encoding='utf-8', errors='replace').splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                folder_num, food_name = parts[0], parts[1].strip()
                name_by_folder[folder_num] = food_name.replace(' ', '_')

    sorted_cat_dirs = sorted(
        [d for d in DATASET_ROOT.iterdir() if d.is_dir() and d.name.isdigit()],
        key=lambda p: int(p.name),
    )

    cls_index = 0
    for cat_dir in sorted_cat_dirs:
        # Resolve food name
        if cat_dir.name in name_by_folder:
            name = name_by_folder[cat_dir.name]
        else:
            per_folder_cat = cat_dir / 'category.txt'
            if per_folder_cat.exists():
                name = per_folder_cat.read_text(encoding='utf-8', errors='replace').strip().replace(' ', '_')
            else:
                name = cat_dir.name

        if name in SKIP_CLASSES:
            skipped_class_names.append(name)
            cat_id_map[cat_dir.name] = -1   # sentinel: skip this folder
        else:
            class_names.append(name)
            cat_id_map[cat_dir.name] = cls_index
            cls_index += 1

    print(f'Classes kept    : {len(class_names)}')
    print(f'Classes skipped : {len(skipped_class_names)}')
    print(f'Sample names    : {class_names[:8]}')

    # ── Convert bb_info.txt → YOLO labels ──────────────────────────────────
    random.seed(42)
    VAL_SPLIT        = 0.15
    skipped_missing  = 0
    skipped_corrupt  = 0
    skipped_degen    = 0

    for cat_dir in sorted_cat_dirs:
        cls_idx = cat_id_map.get(cat_dir.name)
        if cls_idx is None or cls_idx == -1:
            continue   # folder not in category list, or explicitly skipped

        bb_file = cat_dir / 'bb_info.txt'
        if not bb_file.exists():
            continue

        lines = bb_file.read_text(encoding='utf-8', errors='replace').splitlines()

        # Collect all boxes per image (skip the header row)
        boxes_by_img: dict[str, list[tuple[int, int, int, int]]] = {}
        for line in lines[1:]:
            parts = line.split()
            if len(parts) < 5:
                continue
            img_id = parts[0]
            try:
                x1, y1, x2, y2 = int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4])
            except ValueError:
                continue
            if x2 <= x1 or y2 <= y1:
                skipped_degen += 1
                continue
            boxes_by_img.setdefault(img_id, []).append((x1, y1, x2, y2))

        for img_id, boxes in boxes_by_img.items():
            img_path = cat_dir / f'{img_id}.jpg'
            if not img_path.exists():
                skipped_missing += 1
                continue

            try:
                with Image.open(img_path) as im:
                    W, H = im.size
            except Exception:
                skipped_corrupt += 1
                continue

            if W == 0 or H == 0:
                skipped_corrupt += 1
                continue

            split = 'val' if random.random() < VAL_SPLIT else 'train'
            unique_stem = f'{cat_dir.name}_{img_id}'   # avoid cross-category collisions
            dst_img = YOLO_LOCAL / 'images' / split / f'{unique_stem}.jpg'
            dst_lbl = YOLO_LOCAL / 'labels' / split / f'{unique_stem}.txt'

            shutil.copy2(img_path, dst_img)

            yolo_lines = []
            for (x1, y1, x2, y2) in boxes:
                cx = ((x1 + x2) / 2.0) / W
                cy = ((y1 + y2) / 2.0) / H
                w  = (x2 - x1) / W
                h  = (y2 - y1) / H
                # Clamp to [0, 1] in case coords slightly exceed image bounds
                cx = max(0.0, min(1.0, cx))
                cy = max(0.0, min(1.0, cy))
                w  = max(0.0, min(1.0, w))
                h  = max(0.0, min(1.0, h))
                if w <= 0 or h <= 0:
                    skipped_degen += 1
                    continue
                yolo_lines.append(f'{cls_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

            if yolo_lines:
                dst_lbl.write_text('\n'.join(yolo_lines) + '\n')
            else:
                dst_img.unlink(missing_ok=True)

    n_train = len(list((YOLO_LOCAL / 'images' / 'train').glob('*.jpg')))
    n_val   = len(list((YOLO_LOCAL / 'images' / 'val').glob('*.jpg')))

    print(f'\nConversion complete.')
    print(f'  Train images    : {n_train}')
    print(f'  Val images      : {n_val}')
    print(f'  Skipped missing : {skipped_missing}')
    print(f'  Skipped corrupt : {skipped_corrupt}')
    print(f'  Degenerate boxes: {skipped_degen}')

    # ── Copy converted dataset to Drive for future sessions ────────────────
    print(f'\nCopying dataset to Drive — this may take 10-15 minutes...')
    shutil.copytree(
        str(YOLO_LOCAL),
        str(YOLO_DRIVE),
        dirs_exist_ok=True,
    )
    print(f'Dataset saved to: {YOLO_DRIVE}')

    YOLO_ROOT = YOLO_DRIVE

print(f'\nYOLO_ROOT set to: {YOLO_ROOT}')

Classes kept    : 194
Classes skipped : 62
Sample names    : ['eels_on_rice', 'pilaf', "chicken-'n'-egg_on_rice", 'pork_cutlet_on_rice', 'beef_curry', 'sushi', 'chicken_rice', 'fried_rice']

Conversion complete.
  Train images    : 19373
  Val images      : 3468
  Skipped missing : 0
  Skipped corrupt : 0
  Degenerate boxes: 0

Copying dataset to Drive — this may take 10-15 minutes...
Dataset saved to: /content/drive/MyDrive/nutriscan/v2/uec_yolo_v2

YOLO_ROOT set to: /content/drive/MyDrive/nutriscan/v2/uec_yolo_v2


## 5 — Write data.yaml

Writes the YOLO dataset config to three locations:
- `YOLO_ROOT/data.yaml` — used by the trainer
- `SAVE_DIR/data_v2.yaml` — Drive backup
- `SAVE_DIR/classes_v2.txt` — one class name per line, for easy inspection

In [5]:
import yaml

data_yaml = {
    'path': str(YOLO_ROOT),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(class_names),
    'names': class_names,
}

# Primary copy — read by the trainer
yaml_path = YOLO_ROOT / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True)

# Drive backup copy
drive_yaml_path = SAVE_DIR / 'data_v2.yaml'
with open(drive_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True)

# Plain-text class list for easy inspection
classes_txt_path = SAVE_DIR / 'classes_v2.txt'
classes_txt_path.write_text('\n'.join(class_names) + '\n')

print(f'data.yaml written to  : {yaml_path}')
print(f'Drive backup          : {drive_yaml_path}')
print(f'Class list            : {classes_txt_path}')
print(f'nc                    = {data_yaml["nc"]}')
print(f'names[:5]             = {class_names[:5]}')

data.yaml written to  : /content/drive/MyDrive/nutriscan/v2/uec_yolo_v2/data.yaml
Drive backup          : /content/drive/MyDrive/nutriscan/v2/data_v2.yaml
Class list            : /content/drive/MyDrive/nutriscan/v2/classes_v2.txt
nc                    = 194
names[:5]             = ['eels_on_rice', 'pilaf', "chicken-'n'-egg_on_rice", 'pork_cutlet_on_rice', 'beef_curry']


## 6 — Train YOLOv8s for 100 Epochs

- **Model:** `yolov8s.pt` (small — better accuracy than nano for 200+ classes)
- **Epochs:** 100 with early stopping (`patience=20`)
- **Image size:** 640 × 640
- **Batch:** 16 (fits A100 comfortably; reduce to 8 if OOM)
- **Output:** `/content/runs/nutriscan_v2/weights/`

In [6]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # downloads pretrained weights automatically

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    name='nutriscan_v2',
    project='/content/runs',
    exist_ok=True,
    patience=20,      # early stopping — stops if mAP50 does not improve for 20 epochs
    cache=True,       # cache images in RAM for faster training
    device=0,         # GPU
)

print('Training complete.')
print(f'Best weights : {results.save_dir}/weights/best.pt')
print(f'Last weights : {results.save_dir}/weights/last.pt')

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/nutriscan/v2/uec_yolo_v2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=nutriscan_v2, nbs=64, nms=False, opset=None, optimize=False, opti

## 7 — Save Everything to Google Drive

Copies weights and metrics to `MyDrive/nutriscan/v2/` so they persist after the
Colab session ends.

| File | Drive path |
|------|------------|
| Best weights | `v2/best_v2.pt` |
| Last weights | `v2/last_v2.pt` |
| Training metrics | `v2/results_v2.csv` |

In [7]:
import shutil
from pathlib import Path
import pandas as pd

run_dir = Path(results.save_dir)

# ── Copy weights ───────────────────────────────────────────────────────────
best_src = run_dir / 'weights' / 'best.pt'
last_src = run_dir / 'weights' / 'last.pt'
csv_src  = run_dir / 'results.csv'

best_dst = SAVE_DIR / 'best_v2.pt'
last_dst = SAVE_DIR / 'last_v2.pt'
csv_dst  = SAVE_DIR / 'results_v2.csv'

shutil.copy2(best_src, best_dst)
shutil.copy2(last_src, last_dst)

saved_files = [
    ('best_v2.pt',    best_dst),
    ('last_v2.pt',    last_dst),
]

if csv_src.exists():
    shutil.copy2(csv_src, csv_dst)
    saved_files.append(('results_v2.csv', csv_dst))

# ── Print final summary ────────────────────────────────────────────────────
print('=' * 60)
print('SAVE SUMMARY')
print('=' * 60)

# Pull final mAP50 from results.csv
final_map50 = None
if csv_src.exists():
    try:
        df = pd.read_csv(csv_src)
        df.columns = df.columns.str.strip()
        # Column name varies by ultralytics version
        map50_col = next(
            (c for c in df.columns if 'map50' in c.lower() and '95' not in c.lower()),
            None
        )
        if map50_col:
            final_map50 = df[map50_col].max()
    except Exception as e:
        print(f'Could not parse results.csv: {e}')

if final_map50 is not None:
    print(f'Best mAP50 achieved : {final_map50:.4f}')
else:
    print('Best mAP50          : (run validation cell for full metrics)')

print()
print('Saved files:')
for label, path in saved_files:
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f'  {label:<20}  {size_mb:>7.2f} MB   {path}')
print('=' * 60)

SAVE SUMMARY
Best mAP50 achieved : 0.8602

Saved files:
  best_v2.pt              21.61 MB   /content/drive/MyDrive/nutriscan/v2/best_v2.pt
  last_v2.pt              21.61 MB   /content/drive/MyDrive/nutriscan/v2/last_v2.pt
  results_v2.csv           0.01 MB   /content/drive/MyDrive/nutriscan/v2/results_v2.csv


## 8 — Validation & Per-Class mAP50 Report

Loads `best_v2.pt` from Drive and runs a full validation pass.

Reports:
- Overall mAP50 and mAP50-95
- Top 10 best-performing classes
- Top 10 worst-performing classes
- Count of classes above 0.80 mAP50
- Count of classes above 0.70 mAP50

This cell can be run standalone in a fresh session — it re-mounts Drive if needed
and does not depend on earlier cells having been run.

In [8]:
import os
from pathlib import Path

# ── Re-mount Drive if this cell is run in a fresh session ─────────────────
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

SAVE_DIR   = Path('/content/drive/MyDrive/nutriscan/v2')
YOLO_DRIVE = SAVE_DIR / 'uec_yolo_v2'
best_dst   = SAVE_DIR / 'best_v2.pt'
yaml_path  = YOLO_DRIVE / 'data.yaml'

if not best_dst.exists():
    raise FileNotFoundError(f'best_v2.pt not found at {best_dst} — run cells 6-7 first.')
if not yaml_path.exists():
    raise FileNotFoundError(f'data.yaml not found at {yaml_path} — run cells 4-5 first.')

# ── Load model and run validation ─────────────────────────────────────────
from ultralytics import YOLO
import numpy as np

best_model = YOLO(str(best_dst))
metrics = best_model.val(
    data=str(yaml_path),
    imgsz=640,
    device=0,
)

# ── Overall metrics ───────────────────────────────────────────────────────
overall_map50    = metrics.box.map50
overall_map5095  = metrics.box.map

print('=' * 60)
print('VALIDATION RESULTS')
print('=' * 60)
print(f'mAP50            : {overall_map50:.4f}')
print(f'mAP50-95         : {overall_map5095:.4f}')

# ── Per-class mAP50 ───────────────────────────────────────────────────────
# metrics.box.ap_class_index contains the class indices that were evaluated.
# metrics.box.ap50 contains the per-class AP50 values.
import yaml
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)
all_names = data_cfg['names']   # list indexed by class id

class_indices = metrics.box.ap_class_index   # numpy array of class ids
ap50_values   = metrics.box.ap50             # numpy array of AP50 per class

# Build (name, ap50) pairs
per_class = [
    (all_names[int(idx)], float(ap))
    for idx, ap in zip(class_indices, ap50_values)
]
per_class.sort(key=lambda x: x[1], reverse=True)

# Top 10 best
print()
print('Top 10 best-performing classes (mAP50):')
print(f'  {"Class":<35}  mAP50')
print(f'  {"-"*35}  -----')
for name, ap in per_class[:10]:
    print(f'  {name:<35}  {ap:.4f}')

# Top 10 worst
print()
print('Top 10 worst-performing classes (mAP50):')
print(f'  {"Class":<35}  mAP50')
print(f'  {"-"*35}  -----')
for name, ap in per_class[-10:][::-1]:
    print(f'  {name:<35}  {ap:.4f}')

# Threshold counts
above_80 = sum(1 for _, ap in per_class if ap >= 0.80)
above_70 = sum(1 for _, ap in per_class if ap >= 0.70)
total    = len(per_class)

print()
print(f'Classes >= 0.80 mAP50 : {above_80} / {total}  ({100*above_80/total:.1f}%)')
print(f'Classes >= 0.70 mAP50 : {above_70} / {total}  ({100*above_70/total:.1f}%)')
print('=' * 60)

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 73 layers, 11,200,662 parameters, 0 gradients, 28.9 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.0 ms, read: 95.0±60.0 MB/s, size: 149.7 KB)
val: Scanning /content/drive/MyDrive/nutriscan/v2/uec_yolo_v2/labels/val.cache... 3468 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3468/3468 909.1Mit/s 0.0s
val: /content/drive/MyDrive/nutriscan/v2/uec_yolo_v2/images/val/35_3394.jpg: corrupt JPEG restored and saved
val: /content/drive/MyDrive/nutriscan/v2/uec_yolo_v2/images/val/43_4232.jpg: corrupt JPEG restored and saved
val: /content/drive/MyDrive/nutriscan/v2/uec_yolo_v2/images/val/52_5251.jpg: corrupt JPEG restored and saved
val: /content/drive/MyDrive/nutriscan/v2/uec_yolo_v2/images/val/95_12716.jpg: corrupt JPEG restored and saved
val: /content/drive/MyDrive/nutriscan/v2/uec_yolo_v2/images/val/99_82.jpg: corrupt JPEG restored and saved
                 Class     Imag